In [0]:
from pyspark.sql import functions as F

base = "/Volumes/workspace/taxi_weather/raw"

stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", f"{base}/_schemas/taxi")
    .load(f"{base}/landing/taxi")
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path")))

(stream.writeStream
    .option("checkpointLocation", f"{base}/_checkpoints/taxi")
    .trigger(availableNow=True)
    .toTable("workspace.raw.taxi_trips_stream")
    .awaitTermination())

print(spark.table("workspace.raw.taxi_trips_stream").count())
display(spark.table("workspace.raw.taxi_trips_stream").groupBy("_source_file").count())

21932


_source_file,count
/Volumes/workspace/taxi_weather/raw/landing/taxi/feb/part-00000-tid-6648245302725810436-b7bfdb1c-257e-4a25-a9db-90485b507465-129-1.c000.snappy.parquet,11065
/Volumes/workspace/taxi_weather/raw/landing/taxi/jan/part-00000-tid-634060556701997397-296b2a5b-aa3b-481f-a0e3-0c162b4f47a7-128-1.c000.snappy.parquet,10867
